In [16]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

In [17]:
df = pd.read_csv("Ames Housing/AmesHousing-featured.csv")

In [18]:
# 1. Prepare Data
# X = all features, y = Target (SalePrice)
X = df.drop('SalePrice', axis=1)
y = df['SalePrice']

In [19]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [21]:
# 2. Fit on TRAINING data only, then transform both
# This prevents "Data Leakage" from the test set
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [22]:
# 3. Baseline Model (Simple Linear Regression)
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
lr_preds = lr_model.predict(X_test)

In [23]:
# 4. Advanced Model (Ridge Regression with Cross-Validation)
# Ridge helps prevent overfitting when you have many columns
ridge_model = RidgeCV(alphas=(0.1, 1.0, 10.0, 100.0))
ridge_model.fit(X_train_scaled, y_train)
ridge_preds = ridge_model.predict(X_test_scaled)



In [24]:
# 5. Evaluation
def evaluate(y_true, y_pred, name):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    print(f"--- {name} Results ---")
    print(f"RMSE (Log Scale): {rmse:.4f}")
    print(f"R² Score:         {r2:.4f}\n")

In [25]:
evaluate(y_test, lr_preds, "Linear Regression")
evaluate(y_test, ridge_preds, "Ridge Regression")


--- Linear Regression Results ---
RMSE (Log Scale): 0.0997
R² Score:         0.9446

--- Ridge Regression Results ---
RMSE (Log Scale): 0.0975
R² Score:         0.9471



In [34]:
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler

# 1. Scaling is REQUIRED for Lasso to work properly
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 2. Train Lasso with Cross-Validation (finds the best alpha automatically)
lasso_model = LassoCV(eps=0.001, n_alphas=100, cv=5, max_iter=10000)
lasso_model.fit(X_train_scaled, y_train)

# 3. Predict and Evaluate
lasso_preds = lasso_model.predict(X_test_scaled)
rmse = np.sqrt(mean_squared_error(y_test, lasso_preds))
print(f"Lasso RMSE (Log): {rmse:.4f}")

# 4. Feature Selection Analysis
lasso_coefs = pd.Series(lasso_model.coef_, index=X.columns)
kept_count = (lasso_coefs != 0).sum()
dropped_count = (lasso_coefs == 0).sum()

print(f"\nFeatures Kept: {kept_count}")
print(f"Features Dropped: {dropped_count}")

# 5. Top 10 Features
print("\n--- Top 10 Features ---")
print(lasso_coefs.abs().sort_values(ascending=False).head(10))


C:\Users\kkp18\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:1663: FutureWarning: 'n_alphas' was deprecated in 1.7 and will be removed in 1.9. 'alphas' now accepts an integer value which removes the need to pass 'n_alphas'. The default value of 'alphas' will change from None to 100 in 1.9. Pass an explicit value to 'alphas' and leave 'n_alphas' to its default value to silence this warning.
  warnings.warn(


Lasso RMSE (Log): 0.0958

Features Kept: 109
Features Dropped: 115

--- Top 10 Features ---
Gr Liv Area              0.088129
Overall Qual             0.087091
Total SF                 0.063508
House Age                0.044990
Overall Cond             0.041454
BsmtFin SF 1             0.027659
Functional_Sal           0.024863
Sale Condition_Normal    0.024590
Sale Type_New            0.020167
Neighborhood_Crawfor     0.018768
dtype: float64


In [54]:
# 1. Identify the Top 24 features from your Lasso coefficients
top_24_features = lasso_coefs.abs().sort_values(ascending=False).head(52).index
print("--- Selected Top 24 Features ---")
print(list(top_24_features))

# 2. Create a new, reduced dataset
X_train_final = X_train[top_24_features]
X_test_final = X_test[top_24_features]

# 3. Scaling is still recommended for consistency
scaler_final = StandardScaler()
X_train_final_scaled = scaler_final.fit_transform(X_train_final)
X_test_final_scaled = scaler_final.transform(X_test_final)

# 4. Train a Final Ridge Model on these 24 features
# We use Ridge because it's stable and handles the reduced set perfectly
final_model = RidgeCV(alphas=(0.1, 1.0, 10.0))
final_model.fit(X_train_final_scaled, y_train)

# 5. Final Predictions and Evaluation
final_preds = final_model.predict(X_test_final_scaled)

# Using our previously defined evaluate function
evaluate(y_test, final_preds, "Lean Model (Top 24 Features)")


--- Selected Top 24 Features ---
['Gr Liv Area', 'Overall Qual', 'Total SF', 'House Age', 'Overall Cond', 'BsmtFin SF 1', 'Functional_Sal', 'Sale Condition_Normal', 'Sale Type_New', 'Neighborhood_Crawfor', 'Bsmt Exposure', 'MS Zoning_RM', 'MS Zoning_C (all)', 'Kitchen Qual', 'Lot Area', 'Garage Cars', 'Neighborhood_NridgHt', 'Total Bathrooms', 'Central Air_Y', 'Neighborhood_GrnHill', 'Years Since Remodel', 'Screen Porch', 'Condition 1_Norm', 'Foundation_PConc', 'Heating QC', 'Neighborhood_MeadowV', 'Bldg Type_Twnhs', 'MS SubClass', 'Paved Drive_Y', 'Garage Yr Blt', 'Fireplaces', 'Exterior 1st_BrkFace', 'Neighborhood_StoneBr', 'Neighborhood_Somerst', 'Functional_Sev', 'Sale Condition_Partial', 'Bsmt Qual', 'Lot Frontage', 'Heating_Grav', 'Exterior 1st_PreCast', 'Wood Deck SF', 'Functional_Maj2', 'Functional_Typ', 'BsmtFin Type 1', 'Mas Vnr Type_CBlock', 'Bldg Type_Duplex', 'Sale Condition_AdjLand', 'Neighborhood_BrkSide', 'Garage Area', 'Enclosed Porch', 'Garage Cond', 'Open Porch SF']
